# Step 2 — Obtain & Inspect the Public EEG Dataset

**Dataset:** BNCI2014-008 (P300 Speller, first choice for pipeline development)  
**Source:** MOABB (Mother of All BCI Benchmarks)  
**Goal:** Download, inspect metadata, plot raw EEG, verify units, and check event counts.

## 0. Environment

Run this notebook in the project environment created from `Pipfile` and `Pipfile.lock`. Dependencies are not installed from notebook cells.


## 1. Download Dataset via MOABB

MOABB handles caching automatically — the first run downloads to `~/mne_data/`.  
Subsequent runs load from cache.

In [1]:
import moabb
from moabb.datasets import BNCI2014_008
import mne
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from datetime import date

moabb.set_log_level("WARNING")  # suppress verbose MOABB logs
mne.set_log_level("WARNING")

dataset = BNCI2014_008()

print("=" * 50)
print("DATASET INSPECTION RECORD")
print("=" * 50)
print(f"Dataset name    : BNCI2014-008")
print(f"Source          : MOABB (moabb.datasets.BNCI2014_008)")
print(f"Inspection date : {date.today()}")
print(f"MOABB version   : {moabb.__version__}")
print(f"MNE version     : {mne.__version__}")

DATASET INSPECTION RECORD
Dataset name    : BNCI2014-008
Source          : MOABB (moabb.datasets.BNCI2014_008)
Inspection date : 2026-10-22
MOABB version   : 1.7.2
MNE version     : 1.13.2


## 2. Inspect Metadata

Print number of subjects, channels, sampling rate, events, trial counts, and channel names.

In [2]:
import mne
import numpy as np

# Load the dataset definition, but request data for only one subject.
subject_id = dataset.subject_list[0] ## First Subject
subject_data = dataset.get_data(subjects=[subject_id])[subject_id]

print("=" * 60)
print("BNCI2014-008: ONE-SUBJECT INSPECTION")
print("=" * 60)
print(f"Subjects listed in dataset: {len(dataset.subject_list)}")
print(f"Subject loaded: {subject_id}")
print(f"Sessions available for this subject: {len(subject_data)}")

# Select one session.
session_key = next(iter(subject_data))
session_data = subject_data[session_key]

print(f"\nSession selected: {session_key}")
print(f"Runs available in this session: {len(session_data)}")

# Select one run.
run_key = next(iter(session_data))
raw = session_data[run_key]
info = raw.info

print(f"\nRun selected: {run_key}")
print(f"Data type: {type(raw).__name__}")
print(f"Channel count: {raw.info['nchan']}")
print(f"Channel names: {raw.ch_names}")
print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"Recording duration: {raw.times[-1]:.1f} seconds")

# Count events in this run.
events, event_id = mne.events_from_annotations(raw, verbose=False)

print(f"\nEvent mapping: {event_id}")
print(f"Total events: {len(events)}")
for label, code in event_id.items():
    count = int(np.sum(events[:, 2] == code))
    print(f"{label}: {count}")



BNCI2014-008: ONE-SUBJECT INSPECTION
Subjects listed in dataset: 8
Subject loaded: 1
Sessions available for this subject: 1

Session selected: 0
Runs available in this session: 1

Run selected: 0
Data type: RawArray
Channel count: 10
Channel names: ['Fz', 'Cz', 'Pz', 'Oz', 'P3', 'P4', 'PO7', 'PO8', 'Target stim', 'Flash stim']
Sampling rate: 256.0 Hz
Recording duration: 1358.2 seconds

Event mapping: {'NonTarget': 1, 'Target': 2}
Total events: 4200
NonTarget: 3500
Target: 700


/Users/ziadelsisi/.local/share/virtualenvs/eeg-p300-recognition-assessment-gan_qdjr/lib/python3.11/site-packages/moabb/datasets/bnci/base.py:494: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")


In [3]:

print(f"Selected subject: {subject_id}")
print(f"Sessions for this subject: {len(subject_data)}")

for session_key, runs in subject_data.items():
    print(f"\nSession {session_key}: {len(runs)} run(s)")

    for run_key, raw in runs.items():
        print(
            f"  Run {run_key}: {type(raw).__name__}, "
            f"{raw.info['nchan']} channels, "
            f"{raw.info['sfreq']} Hz, "
            f"{raw.times[-1]:.1f} seconds"
        )

Selected subject: 1
Sessions for this subject: 1

Session 0: 1 run(s)
  Run 0: RawArray, 10 channels, 256.0 Hz, 1358.2 seconds


## 3. Plot Raw EEG (10–20 seconds)

In [4]:
%matplotlib inline
import matplotlib.pyplot as plt
import mne
from pathlib import Path

# Plot seconds 5–20 from the selected run.
t_start, t_end = 5.0, 20.0
sfreq = raw.info["sfreq"]

start_sample = int(t_start * sfreq)
end_sample = min(int(t_end * sfreq), raw.n_times)

# Exclude stimulus channels so they don't distort the EEG plot scale.
eeg_picks = mne.pick_types(raw.info, eeg=True, stim=False)
eeg_names = [raw.ch_names[i] for i in eeg_picks]
data = raw.get_data(picks=eeg_picks, start=start_sample, stop=end_sample) * 1e6
times = raw.times[start_sample:end_sample]

fig, axes = plt.subplots(
    len(eeg_picks), 1,
    figsize=(14, 1.5 * len(eeg_picks)),
    sharex=True,
    squeeze=False,
)

for i, ax in enumerate(axes[:, 0]):
    ax.plot(times, data[i], lw=0.7, color="steelblue")
    ax.set_ylabel(eeg_names[i], rotation=0, labelpad=35, va="center")
    ax.axhline(0, color="grey", lw=0.4, ls="--")

axes[-1, 0].set_xlabel("Time (s)")
fig.suptitle(
    f"Raw EEG — Subject {subject_id} | Session {session_key} | "
    f"Run {run_key} | {t_start}–{t_end} s"
)
plt.tight_layout()

# Save the plot and related inspection artifacts under one results folder.
project_root = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "Pipfile").exists()),
    Path.cwd(),
)
results_dir = project_root / "results" / "BNCI2014-008-Dataset-Inspection"
figures_dir = project_root / "figures" / "BNCI2014-008-Dataset-Inspection"
plot_path = figures_dir / "raw_eeg_plot.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")

plt.show()

<ipython-input-4-826f8bc34538>:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Check Units

MOABB/MNE loads EEG in **Volts (V)** internally. We scale to µV only for display.

In [5]:
# ── Unit check ────────────────────────────────────────────────────────────────
import mne

eeg_picks = mne.pick_types(raw.info, eeg=True, stim=False)
if len(eeg_picks) == 0:
    raise ValueError("No EEG channels found in the selected run.")

sample_index = int(10 * raw.info["sfreq"])  # sample at 10 seconds
sample_value_V = raw.get_data(
    picks=[eeg_picks[0]],
    start=sample_index,
    stop=sample_index + 1,
)[0, 0]
sample_value_uV = sample_value_V * 1e6

print("=" * 50)
print("UNIT HANDLING RECORD")
print("=" * 50)
print("MNE EEG data unit       : Volts (V)")
print(f"Sample from raw         : {sample_value_V:.6e} V")
print(f"Same sample for display : {sample_value_uV:.4f} µV")
print("Display conversion      : multiply volts by 1e6")

channel_types = dict(zip(raw.ch_names, raw.get_channel_types()))
print("\nChannel types:")
for channel_type in sorted(set(channel_types.values())):
    names = [
        name for name, kind in channel_types.items()
        if kind == channel_type
    ]
    print(f"  {channel_type}: {names}")

UNIT HANDLING RECORD
MNE EEG data unit       : Volts (V)
Sample from raw         : -6.397851e-07 V
Same sample for display : -0.6398 µV
Display conversion      : multiply volts by 1e6

Channel types:
  eeg: ['Fz', 'Cz', 'Pz', 'Oz', 'P3', 'P4', 'PO7', 'PO8']
  stim: ['Target stim', 'Flash stim']


## 5. Check Events — Target vs NonTarget Counts

In [6]:
# ── Extract events from annotations ───────────────────────────────────────────
events, event_id = mne.events_from_annotations(raw)

print("=" * 50)
print("EVENT SUMMARY")
print("=" * 50)
print(f"Event ID mapping: {event_id}")
print()

# BNCI2014-008 uses event IDs: 1 = NonTarget, 2 = Target
# (MOABB maps 'Target' and 'NonTarget' annotations to integers)
for label, code in sorted(event_id.items(), key=lambda x: x[1]):
    count = int((events[:, 2] == code).sum())
    print(f"  Event '{label}' (ID={code}) : {count} trials")

# Identify Target and NonTarget robustly
target_label     = [k for k in event_id if "arget" in k and "Non" not in k]
nontarget_label  = [k for k in event_id if "on" in k.lower() and "arget" in k]

if target_label and nontarget_label:
    n_target    = int((events[:, 2] == event_id[target_label[0]]).sum())
    n_nontarget = int((events[:, 2] == event_id[nontarget_label[0]]).sum())
    ratio       = n_nontarget / n_target if n_target > 0 else float('inf')

    print()
    print("=" * 50)
    print("SECTION 4 FILL-IN — TARGET vs NONTARGET")
    print("=" * 50)
    print(f"Target trials    : {n_target}")
    print(f"NonTarget trials : {n_nontarget}")
    print(f"Class ratio      : 1 Target : {ratio:.1f} NonTarget  (expected ~1:5 for P300)")
else:
    print("\n[Note] Could not auto-identify Target/NonTarget — check event_id labels above.")

EVENT SUMMARY
Event ID mapping: {'NonTarget': 1, 'Target': 2}

  Event 'NonTarget' (ID=1) : 3500 trials
  Event 'Target' (ID=2) : 700 trials

SECTION 4 FILL-IN — TARGET vs NONTARGET
Target trials    : 700
NonTarget trials : 3500
Class ratio      : 1 Target : 5.0 NonTarget  (expected ~1:5 for P300)


## 6. Epoch Quick-Check (sanity)

Epoch around stimulus onset and confirm shape.

In [7]:
# ── Create epochs for a quick sanity check ────────────────────────────────────
tmin, tmax = -0.2, 0.8  # standard P300 window: -200 ms to +800 ms

epochs = mne.Epochs(
    raw, events, event_id=event_id,
    tmin=tmin, tmax=tmax,
    baseline=(tmin, 0),
    preload=True,
    verbose=False
)

print("=" * 50)
print("EPOCH SANITY CHECK")
print("=" * 50)
print(f"Epoch shape (n_epochs, n_channels, n_times) : {epochs.get_data().shape}")
print(f"Time window  : {tmin*1000:.0f} ms to {tmax*1000:.0f} ms")
print(f"Baseline     : {tmin*1000:.0f} ms to 0 ms")
print()
print(epochs)

EPOCH SANITY CHECK
Epoch shape (n_epochs, n_channels, n_times) : (4200, 10, 257)
Time window  : -200 ms to 800 ms
Baseline     : -200 ms to 0 ms

<Epochs | 4200 events (all good), -0.199 – 0.801 s (baseline -0.2 – 0 s), ~82.4 MiB, data loaded,
 'NonTarget': 3500
 'Target': 700>


## 7. Summary Table — Section 4 Fill-in

Collect all evidence in one place.

In [8]:
print("=" * 60)
print("SECTION 4 — INSPECTION SUMMARY")
print("=" * 60)

rows = [
    ("Dataset", "BNCI2014-008"),
    ("Source", "MOABB BNCI2014_008; DOI: 10.3389/fnhum.2013.00732"),
    ("Inspection date", str(date.today())),
    ("Subjects available", str(len(dataset.subject_list))),
    ("Subject inspected", str(subject_id)),
    ("Sessions for subject", str(len(subject_data))),
    ("Session inspected", str(session_key)),
    ("Runs in session", str(len(session_data))),
    ("Run inspected", str(run_key)),
    ("EEG channels", ", ".join(eeg_names)),
    ("Sampling rate", f"{raw.info['sfreq']} Hz"),
    ("Recording duration", f"{raw.times[-1]:.1f} seconds"),
    ("MNE EEG unit", "Volts (V); multiply by 1e6 for µV display"),
    ("Event IDs", str(event_id)),
    ("Target events", str(n_target)),
    ("NonTarget events", str(n_nontarget)),
    ("Target:NonTarget ratio", f"1:{ratio:.1f}"),
    ("Epoch shape", str(epochs.get_data().shape)),
    ("Raw EEG plot", "../../figures/BNCI2014-008-Dataset-Inspection/raw_eeg_plot.png"),
]

for field, value in rows:
    print(f"{field:<28}: {value}")
print("=" * 60)

# Save a portable Markdown report alongside the raw EEG figure.
report_path = results_dir / "inspection_summary.md"
report_rows = "\n".join(
    f"| {field} | {str(value).replace('|', '&#124;')} |"
    for field, value in rows
)
report = (
    "# BNCI2014-008 Dataset Inspection Summary\n\n"
    f"Generated: {date.today()}\n\n"
    "This report summarizes the subject, session, and run inspected by the notebook.\n\n"
    "| Item | Inspection result |\n|---|---|\n"
    f"{report_rows}\n\n"
    "## Raw EEG figure\n\n"
    "![Raw EEG plot](../../figures/BNCI2014-008-Dataset-Inspection/raw_eeg_plot.png)\n"
)
report_path.write_text(report, encoding="utf-8")
print(f"Inspection report saved to {report_path}")


SECTION 4 — INSPECTION SUMMARY
Dataset                     : BNCI2014-008
Source                      : MOABB BNCI2014_008; DOI: 10.3389/fnhum.2013.00732
Inspection date             : 2026-10-22
Subjects available          : 8
Subject inspected           : 1
Sessions for subject        : 1
Session inspected           : 0
Runs in session             : 1
Run inspected               : 0
EEG channels                : Fz, Cz, Pz, Oz, P3, P4, PO7, PO8
Sampling rate               : 256.0 Hz
Recording duration          : 1358.2 seconds
MNE EEG unit                : Volts (V); multiply by 1e6 for µV display
Event IDs                   : {'NonTarget': 1, 'Target': 2}
Target events               : 700
NonTarget events            : 3500
Target:NonTarget ratio      : 1:5.0
Epoch shape                 : (4200, 10, 257)
Raw EEG plot                : ../../figures/BNCI2014-008-Dataset-Inspection/raw_eeg_plot.png
Inspection report saved to /Users/ziadelsisi/Hello World/Projects/Graduation Project /eeg-